# Protein Plots

> Transcript visualization with protein domain overlay

In [ ]:
#| default_exp protein_plots

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from typing import List, Optional, Tuple, Dict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle, Patch, FancyBboxPatch

from allos.transcript_data import TranscriptData
from allos.protein_data import ProteinData
from allos.transcript_plots import TranscriptPlots
import seaborn as sns

In [ ]:
#| export
class ProteinPlots(TranscriptPlots):
    """Extended TranscriptPlots with protein domain overlay capabilities.
    
    Inherits all transcript plotting functionality from TranscriptPlots and adds
    methods for overlaying protein domains on transcript structures.
    """
    
    def __init__(
        self,
        transcript_data: TranscriptData,
        protein_data: ProteinData,
        intron_scale: float = 0.15,
        color_offset: int = 0,
        **kwargs
    ):
        """Initialize with both TranscriptData and ProteinData."""
        super().__init__(transcript_data=transcript_data, intron_scale=intron_scale, color_offset=color_offset, **kwargs)
        self.protein_data = protein_data

    def inspect_protein_features(
        self,
        transcript_id: str,
        provider: Optional[str] = None,
    ) -> Dict:
        """Inspect available protein features for a transcript.
        
        This method shows you what raw features are available from the API
        before filtering, so you can decide which sources or ID prefixes to use.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID to inspect
        provider : str, optional
            Domain provider ("ensembl" or "interpro")
            If None, uses the ProteinData instance default
        
        Returns
        -------
        summary : Dict
            Dictionary with keys:
            - 'feature_types': unique feature types found
            - 'sources': unique source databases
            - 'id_prefixes': common ID prefixes (for filtering)
            - 'features': list of all raw features
            
        Examples
        --------
        >>> summary = pp.inspect_protein_features("ENSMUST00000218127")
        >>> print("Available sources:", summary['sources'])
        >>> print("Available ID prefixes:", summary['id_prefixes'])
        >>> # Now plot with filtered sources
        >>> fig = pp.draw_transcripts_with_protein_domains(
        ...     ["ENSMUST00000218127"],
        ...     sources=["pfam", "smart"]
        ... )
        """
        return self.protein_data.inspect_features(transcript_id, provider)
    
    def preview_protein_domains(
        self,
        transcript_id: str,
        provider: Optional[str] = None,
        max_domains: int = 12,
        id_prefixes: Optional[List[str]] = None,
        sources: Optional[List[str]] = None,
    ) -> List[Dict]:
        """Preview what protein domains will be shown for a transcript.
        
        Use this to see exactly which domains will be displayed before
        generating the full plot. Helps tune filtering parameters.
        
        Parameters
        ----------
        transcript_id : str
            Transcript ID
        provider : str, optional
            Domain provider
        max_domains : int, default 12
            Maximum domains to show
        id_prefixes : List[str], optional
            Filter by ID prefix
        sources : List[str], optional
            Filter by source database
        
        Returns
        -------
        domains : List[Dict]
            List of domain dicts that will be displayed
            Each has keys: name, label, start, end (in AA coordinates)
        
        Examples
        --------
        >>> # See what domains are available
        >>> summary = pp.inspect_protein_features("ENSMUST00000218127")
        >>> print("Sources:", summary['sources'])
        
        >>> # Preview with different filters
        >>> domains_all = pp.preview_protein_domains("ENSMUST00000218127")
        >>> domains_pfam = pp.preview_protein_domains(
        ...     "ENSMUST00000218127",
        ...     sources=["pfam"]
        ... )
        >>> print(f"All sources: {len(domains_all)} domains")
        >>> print(f"Pfam only: {len(domains_pfam)} domains")
        
        >>> # Now plot with chosen filter
        >>> fig = pp.draw_transcripts_with_protein_domains(
        ...     ["ENSMUST00000218127"],
        ...     sources=["pfam"],
        ...     max_domains=6
        ... )
        """
        return self.protein_data.get_protein_domains(
            transcript_id,
            provider=provider,
            max_domains=max_domains,
            id_prefixes=id_prefixes,
            sources=sources,
        )

    

    def survey_gene_protein_domains(
        self,
        gene: str,
        provider: Optional[str] = None,
        max_transcripts: Optional[int] = None,
    ):
        """Survey protein domains to help decide what to plot.
        
        Shows all domain IDs retrieved, grouped by prefix, so you can easily
        filter out uninformative domains (like AF-, mobidb, seg, Coil) and
        focus on the functional domains (PF, SM, PS, cd, SSF, etc.).
        
        Parameters
        ----------
        gene : str
            Gene name or gene ID
        provider : str, optional
            Domain provider ("ensembl" or "interpro")
        max_transcripts : int, optional
            Limit to first N transcripts
        
        Returns
        -------
        summary : dict
            Dictionary with:
            - 'all_domain_ids': set of all unique domain IDs
            - 'prefix_groups': dict mapping prefix to list of domain IDs
            - 'domain_info': dict with descriptions
            - 'transcripts': list of transcript IDs
        
        Examples
        --------
        >>> # Survey to see what domains are available
        >>> survey = pp.survey_gene_protein_domains("Snap25")
        >>> 
        >>> # Based on output, plot with useful prefixes only
        >>> fig = pp.draw_transcripts_with_protein_domains(
        ...     survey['transcripts'],
        ...     id_prefixes=["PF", "SM", "PS", "cd", "SSF"],  # Exclude AF-, mobidb, etc
        ...     max_domains=8
        ... )
        """
        import pandas as pd
        from collections import defaultdict
        
        # Get transcripts
        all_tids = self.transcript_data.get_transcripts_by_gene_id(gene)
        if not all_tids:
            all_tids = self.transcript_data.get_transcripts_by_gene_name(gene)
        
        if not all_tids:
            raise ValueError(f"No transcripts found for gene: {gene}")
        
        if max_transcripts:
            all_tids = all_tids[:max_transcripts]
        
        print("=" * 80)
        print(f"DOMAIN SURVEY: {gene}")
        print("=" * 80)
        print(f"Surveying {len(all_tids)} isoform(s): {', '.join(all_tids[:3])}")
        if len(all_tids) > 3:
            print(f"  ... and {len(all_tids)-3} more")
        
        # Collect all domains across all transcripts
        all_domain_ids = set()
        domain_info = {}  # domain_id -> {desc, source, count}
        
        for tid in all_tids:
            try:
                features = self.protein_data.get_raw_features(tid, provider)
                for feat in features:
                    dom_id = feat.get('id', '')
                    all_domain_ids.add(dom_id)
                    if dom_id not in domain_info:
                        domain_info[dom_id] = {
                            'desc': feat.get('desc', ''),
                            'source': feat.get('source', ''),
                            'count': 0
                        }
                    domain_info[dom_id]['count'] += 1
            except Exception as e:
                print(f"  Warning: Could not get features for {tid}")
        
        if not all_domain_ids:
            print("\nNo domains found!")
            return {
                'all_domain_ids': set(),
                'prefix_groups': {},
                'domain_info': {},
                'transcripts': all_tids
            }
        
        # Group by prefix (2-3 char)
        prefix_groups = defaultdict(list)
        for dom_id in sorted(all_domain_ids):
            dom_str = str(dom_id)
            # Try different prefix lengths
            for prefix_len in [2, 3]:
                if len(dom_str) >= prefix_len:
                    prefix = dom_str[:prefix_len]
                    prefix_groups[prefix].append(dom_id)
        
        # Count unique prefixes and consolidate
        prefix_counts = {}
        for prefix, domains in prefix_groups.items():
            if len(prefix) == 2:
                prefix_counts[prefix] = len(set(domains))
        
        print(f"\n" + "=" * 80)
        print(f"DOMAINS RETRIEVED: {len(all_domain_ids)} unique domain IDs")
        print("=" * 80)
        
        # Show domains grouped by prefix
        print(f"\nDomains by ID prefix (for filtering with id_prefixes=[...]):")
        print("-" * 80)
        
        # Sort prefixes by count (most common first)
        sorted_prefixes = sorted(prefix_counts.items(), key=lambda x: -x[1])
        
        for prefix, count in sorted_prefixes:
            # Get domains with this prefix
            domains_with_prefix = [d for d in all_domain_ids if str(d).startswith(prefix)]
            
            # Show count and examples
            examples = sorted(domains_with_prefix)[:3]
            example_str = ", ".join(str(e) for e in examples)
            if len(domains_with_prefix) > 3:
                example_str += f", ... ({len(domains_with_prefix)-3} more)"
            
            # Categorize as useful or not
            is_junk = prefix in ["AF", "mo", "se", "Co", "1."]
            marker = "  ⚠️ " if is_junk else "  ✓ "
            
            print(f"{marker}'{prefix}*' : {count:3d} domains  ({example_str})")
        
        # Show recommendations
        print(f"\n" + "=" * 80)
        print("FILTERING RECOMMENDATIONS")
        print("=" * 80)
        
        # Identify likely junk prefixes
        junk_prefixes = {"AF", "mo", "se", "Co", "1."}
        useful_prefixes = [p for p, c in sorted_prefixes if p not in junk_prefixes][:6]
        
        print(f"\n⚠️  Typically uninformative (EXCLUDE these):")
        print(f"   AF-* : AlphaFold mappings (not functional domains)")
        print(f"   mo*  : mobidb-lite disorder predictions")
        print(f"   se*  : seg low-complexity regions")
        print(f"   Co*  : Coiled-coil predictions")
        print(f"   1.*  : Generic CATH/Gene3D (entire protein)")
        
        print(f"\n✓  Typically informative (KEEP these):")
        print(f"   PF*  : Pfam protein families")
        print(f"   SM*  : SMART domains")
        print(f"   PS*  : PROSITE patterns")
        print(f"   cd*  : Conserved domains (CDD)")
        print(f"   SS*  : SUPERFAMILY")
        print(f"   PT*  : PANTHER")
        
        print(f"\n" + "=" * 80)
        print("SUGGESTED PLOTTING COMMANDS")
        print("=" * 80)
        
        if useful_prefixes:
            print(f"\n# Option 1: Plot with useful domains only")
            print(f"fig = pp.draw_transcripts_with_protein_domains(")
            print(f"    {all_tids[:3]},")
            print(f"    id_prefixes={useful_prefixes[:5]},  # Filter to functional domains")
            print(f"    max_domains=8")
            print(f")")
            
            print(f"\n# Option 2: Plot specific domain types")
            print(f"fig = pp.draw_transcripts_with_protein_domains(")
            print(f"    {all_tids[:3]},")
            print(f"    id_prefixes=['PF', 'SM'],  # Pfam + SMART only")
            print(f"    max_domains=6")
            print(f")")
        
        print(f"\n# Option 3: See all domains (unfiltered)")
        print(f"fig = pp.draw_transcripts_with_protein_domains({all_tids[:3]}, max_domains=20)")
        
        print("=" * 80)
        
        return {
            'all_domain_ids': all_domain_ids,
            'prefix_groups': dict(prefix_groups),
            'domain_info': domain_info,
            'transcripts': all_tids
        }

    def _draw_linearized_protein_track(
        self,
        ax,
        tid: str,
        exons,
        strand,
        cds_bounds,
        mapper,
        y_center: float,
        doms_aa: list,
        id2color: dict,
        exon_color,
        prot_len: int,
    ):
        """Draw full-width linearized protein track with trapezoidal CDS connectors.

        The protein sequence is mapped linearly across the full plot width (0→1),
        matching transcript orientation (reversed for minus-strand genes).
        Semi-transparent trapezoids link each CDS exon block to its AA range.
        """
        h_exon = self.exon_height
        dom_height = h_exon * 0.65
        is_minus = strand in (-1, "-", "minus")

        # Y positions
        y_exon_top = y_center + h_exon / 2
        connector_gap = h_exon * 0.85
        y_prot_bot = y_exon_top + connector_gap
        y_prot = y_prot_bot + dom_height / 2
        y_prot_top = y_prot_bot + dom_height

        def prot_x(aa):
            """Map 1-based AA position to display x ∈ [0,1]; aa=1→0, aa=prot_len→1."""
            frac = (aa - 1) / max(1, prot_len - 1)
            return (1.0 - frac) if is_minus else frac

        # Build CDS exon segments with cumulative AA ranges
        cs, ce = sorted(cds_bounds)
        ex_sorted = sorted(((min(a, b), max(a, b)) for a, b in exons), key=lambda x: x[0])
        coding_segs = []
        for s, e in ex_sorted:
            ss, se = max(s, cs), min(e, ce)
            if se > ss:
                coding_segs.append({"g_start": ss, "g_end": se, "length": se - ss})

        segs_cds = list(reversed(coding_segs)) if is_minus else coding_segs
        off = 0
        for seg in segs_cds:
            seg["cds_offset"] = off
            off += seg["length"]

        # Draw trapezoidal connectors (CDS exon → protein region)
        for seg in segs_cds:
            aa_s = seg["cds_offset"] / 3 + 1          # 1-based start
            aa_e = (seg["cds_offset"] + seg["length"]) / 3  # end (non-inclusive)
            gx_l = mapper(seg["g_start"])
            gx_r = mapper(seg["g_end"])
            px_l = min(prot_x(aa_s), prot_x(aa_e))
            px_r = max(prot_x(aa_s), prot_x(aa_e))
            verts = [
                (gx_l, y_exon_top), (gx_r, y_exon_top),
                (px_r, y_prot_bot), (px_l, y_prot_bot),
            ]
            ax.add_patch(mpl.patches.Polygon(
                verts, closed=True,
                facecolor=exon_color, edgecolor="none", alpha=0.22, zorder=0.5,
            ))

        # Subtle background panel for protein track
        ax.add_patch(FancyBboxPatch(
            (-0.005, y_prot_bot - 0.003), 1.010, dom_height + 0.006,
            boxstyle="round,pad=0,rounding_size=0.005",
            facecolor="#f5f5f5", edgecolor="#c8c8c8",
            linewidth=0.7, alpha=0.90, zorder=0.9,
        ))

        # Protein track baseline
        ax.add_line(mpl.lines.Line2D(
            [0.0, 1.0], [y_prot, y_prot],
            color="#c0c0c0", linewidth=1.0, zorder=1,
        ))

        # Domain blocks in protein (AA) space
        for dom in doms_aa:
            dom_id = dom["name"]
            col = id2color.get(dom_id, "#444444")
            x_l = min(prot_x(dom["start"]), prot_x(dom["end"]))
            x_r = max(prot_x(dom["start"]), prot_x(dom["end"]))
            width = x_r - x_l
            if width < 5e-4:
                continue
            rounding = min(width * 0.3, dom_height * 0.45, 0.015)
            # Derive a darker edge from the fill color
            import colorsys as _cs
            _cr, _cg, _cb = (col[0], col[1], col[2]) if isinstance(col, (list, tuple)) else (0.4, 0.4, 0.4)
            _h, _s, _v = _cs.rgb_to_hsv(_cr, _cg, _cb)
            _edge = _cs.hsv_to_rgb(_h, min(1.0, _s * 1.15), _v * 0.60)
            ax.add_patch(FancyBboxPatch(
                (x_l, y_prot_bot), width, dom_height,
                boxstyle=f"round,pad=0,rounding_size={rounding}",
                facecolor=col, edgecolor=_edge,
                linewidth=0.5, alpha=0.95, zorder=3,
            ))


        # Inline domain labels — greedy widest-first, text-extent collision check
        # (allows multiple labels even on stacked domains as long as text doesn't collide)
        _CHAR_W = 0.0033        # estimated data-units per char at 6.5 pt
        _PAD    = 0.010         # horizontal padding around each label
        _SKIP_IDS   = ("AF-", "1.")
        _SKIP_WORDS = ("mapped using", "disorder", "seg", "coil")

        _dom_rects = []
        for dom in doms_aa:
            _xl2 = min(prot_x(dom["start"]), prot_x(dom["end"]))
            _xr2 = max(prot_x(dom["start"]), prot_x(dom["end"]))
            _dom_rects.append((dom, _xl2, _xr2, _xr2 - _xl2))

        _placed = []  # list of (center_x, half_text_width) for placed labels
        for dom, _xl2, _xr2, _w in sorted(_dom_rects, key=lambda t: -t[3]):
            if _w < 0.045:
                continue
            raw = (dom.get("label") or dom["name"]).strip()
            _dom_id = dom.get("name", "")
            if (_dom_id.startswith(_SKIP_IDS)
                    or any(raw.lower().startswith(p) for p in _SKIP_WORDS)
                    or raw.lower() in ("seg", "coil", "disorder_prediction")):
                continue
            words = raw.split()
            if len(raw) <= 15:
                short = raw
            elif len(words) >= 2 and len(words[0]) + 1 + len(words[1]) <= 15:
                short = f"{words[0]} {words[1]}"
            else:
                short = words[0][:15]
            _cx   = _xl2 + _w / 2
            _half = len(short) * _CHAR_W / 2 + _PAD
            # Skip if text extent overlaps any already-placed label
            if any(abs(_cx - pc) < _half + ph for pc, ph in _placed):
                continue
            # Also skip if text would overflow the domain block on either side
            if _cx - _half < _xl2 - 0.005 or _cx + _half > _xr2 + 0.005:
                continue
            _placed.append((_cx, _half))
            ax.text(
                _cx, y_prot_bot + dom_height / 2,
                short, ha="center", va="center",
                fontsize=6.5, color="white", fontweight="bold",
                clip_on=True, zorder=5,
            )

        # Exon boundary ticks below protein track panel
        _seen_px = set()
        for seg in segs_cds:
            for _aa in [seg["cds_offset"] / 3 + 1,
                        (seg["cds_offset"] + seg["length"]) / 3]:
                _px = round(prot_x(_aa), 6)
                if _px in _seen_px or _px <= 0.002 or _px >= 0.998:
                    continue
                _seen_px.add(_px)
                ax.add_line(mpl.lines.Line2D(
                    [_px, _px], [y_prot_bot - 0.012, y_prot_bot],
                    color="#999999", linewidth=0.9, zorder=4,
                ))

        # AA ruler above protein track
        y_rule = y_prot_top + 0.010
        major_step = 25 if prot_len <= 150 else 50 if prot_len <= 400 else 100 if prot_len <= 800 else 200
        minor_step = max(5, major_step // 5)

        # Ruler baseline
        ax.add_line(mpl.lines.Line2D(
            [0.0, 1.0], [y_rule, y_rule],
            color="#cccccc", linewidth=0.5, zorder=2,
        ))

        # Minor ticks
        for aa in range(minor_step, prot_len, minor_step):
            if aa % major_step == 0:
                continue
            ax.add_line(mpl.lines.Line2D(
                [prot_x(aa), prot_x(aa)], [y_rule, y_rule + 0.008],
                color="#bbbbbb", linewidth=0.4, zorder=2,
            ))

        # Major ticks + labels
        major_ticks = list(range(1, prot_len + 1, major_step))
        if prot_len not in major_ticks:
            major_ticks.append(prot_len)
        for aa in major_ticks:
            px = prot_x(aa)
            ax.add_line(mpl.lines.Line2D(
                [px, px], [y_rule, y_rule + 0.018],
                color="#666666", linewidth=0.8, zorder=2,
            ))
            ax.text(px, y_rule + 0.022, str(aa),
                    ha="center", va="bottom", fontsize=8,
                    color="#555555", family="monospace")

    def draw_transcripts_with_protein_domains(
        self,
        transcript_ids,
        *,
        provider: str = "ensembl",
        max_domains: int = 12,
        id_prefixes: Optional[List[str]] = None,
        sources: Optional[List[str]] = None,
        fig_width: float = 18.0,
        palette_name: str = "Dark2",
        custom_domains: Optional[Dict[str, List[Dict]]] = None,
        show_aa_ticks: bool = True,
        show_linearized_protein: bool = True,
    ):
        """Draw isoform exon structures + clean domain tracks.
        
        For InterPro provider: domains are only shown on the first (canonical) transcript,
        since InterPro annotations are only available for canonical isoforms.
        Other transcripts are still shown for structural comparison.
        
        Parameters
        ----------
        transcript_ids : str or List[str]
            Transcript ID(s) to visualize
        provider : str, default "ensembl"
            Domain provider ("ensembl" or "interpro")
            Note: InterPro only annotates canonical transcripts
        max_domains : int, default 12
            Maximum domains to show per transcript
        id_prefixes : List[str], optional
            Filter domains by ID prefix (e.g., ["PF"] for Pfam)
        sources : List[str], optional
            Filter domains by source database
        fig_width : float, default 18.0
            Figure width in inches
        palette_name : str, default "Dark2"
            Seaborn color palette name for protein domains
        custom_domains : Dict[str, List[Dict]], optional
            Custom domain annotations. Dict mapping transcript_id to list of domain dicts.
            Each domain dict should have keys: 'name', 'label', 'start', 'end' (AA coords)
            If provided, overrides API fetching for those transcripts.
        show_aa_ticks : bool, default True
            Show amino acid position ticks on domain track (only used when
            show_linearized_protein=False)
        show_linearized_protein : bool, default True
            Replace the compressed genomic domain track with a full-width
            linearized protein view connected to CDS exons by trapezoid fills.
        
        Returns
        -------
        fig : matplotlib.figure.Figure
            The generated figure
        
        Examples
        --------
        # Use custom domains for specific transcripts
        >>> custom = {
        ...     "ENST00000317610": [
        ...         {"name": "CustomDomain1", "label": "My Domain", "start": 10, "end": 50}
        ...     ]
        ... }
        >>> fig = pp.draw_transcripts_with_protein_domains(
        ...     ["ENST00000317610"],
        ...     custom_domains=custom
        ... )
        """
        if isinstance(transcript_ids, str):
            tids = [transcript_ids]
        else:
            tids = list(transcript_ids)
        if not tids:
            raise ValueError("No transcript IDs given.")
        
        # Get layout and mapping info from parent class
        (
            tids_layout,
            exons_list,
            strands,
            cds_map,
            mapper,
            (g0, g1),
            exonic_len,
            genomic_span,
        ) = self._layout_and_mapper(tids, draw_cds=True)
        
        n_rows = len(tids_layout)
        
        def _filter_features(features):
            fs = features
            if sources is not None:
                fs = [f for f in fs if f.get("source") in sources]
            if id_prefixes is not None:
                fs = [
                    f for f in fs
                    if any(str(f.get("id", "")).startswith(p) for p in id_prefixes)
                ]
            return fs
        
        # Determine which transcripts to fetch domains for
        if custom_domains:
            # If custom domains provided, use those
            tids_for_domains = tids_layout
        elif provider.lower() == "interpro":
            # For InterPro, only fetch for first (canonical) transcript
            tids_for_domains = [tids_layout[0]] if tids_layout else []
        else:
            # For Ensembl, fetch for all transcripts
            tids_for_domains = tids_layout
        
        # Collect domains per isoform + global IDs/labels
        doms_per_tid = {}
        all_dom_ids = []
        id2label = {}
        
        for tid in tids_for_domains:
            # Check if custom domains provided for this transcript
            if custom_domains and tid in custom_domains:
                domains_aa = custom_domains[tid]
            else:
                # Fetch from API
                domains_aa = self.protein_data.get_protein_domains(
                    tid,
                    provider=provider,
                    max_domains=max_domains,
                    id_prefixes=id_prefixes,
                    sources=sources,
                )
            
            if not domains_aa:
                doms_per_tid[tid] = []
                continue
            
            doms_per_tid[tid] = domains_aa
            
            for d in domains_aa:
                dom_id = d["name"]
                all_dom_ids.append(dom_id)
                id2label.setdefault(dom_id, d.get("label") or dom_id)
        
        # Assign colors to unique domain IDs
        uniq_dom_ids = sorted(set(all_dom_ids))
        # Use husl when n_doms exceeds the named palette capacity, ensuring no repeating colors
        n_doms = len(uniq_dom_ids)
        _base_palette = sns.color_palette(palette_name)
        dom_colors_list = sns.color_palette("husl", n_doms) if n_doms > len(_base_palette) else sns.color_palette(palette_name, n_doms)
        id2color = {dom_id: dom_colors_list[i] for i, dom_id in enumerate(uniq_dom_ids)}
        
        # Create figure + axes  (wider row pitch for protein layout)
        _orig_pitch = self.row_pitch
        if show_linearized_protein:
            self.row_pitch = 0.88
        fig, ax, ycs = self._prep_axes(
            n_rows,
            reserve_panel_space=False,
            fig_width=fig_width,
            extra_top=0.85,
        )
        if show_linearized_protein and n_rows > 0:
            fig.set_size_inches(fig_width, max(3.5, 3.0 * n_rows))
        self.row_pitch = _orig_pitch
        
        exon_cols = getattr(self, "colors", sns.color_palette("pastel", n_rows))
        h_exon = self.exon_height
        dom_height = h_exon * 0.50
        
        # Draw each transcript with its domain track
        for row_idx, (tid, exons, strand, y_center) in enumerate(
            zip(tids_layout, exons_list, strands, ycs)
        ):
            exon_color = exon_cols[row_idx % len(exon_cols)]
            self._draw_one(
                exons,
                strand,
                exon_color,
                tid,
                mapper,
                y_center=y_center,
                cds_bounds=cds_map[tid],
                show_row_bounds=False,
                length_bar=None,
            )
            
            # Get domains and CDS bounds for this transcript
            doms_aa = doms_per_tid.get(tid) or []
            cds_bounds = cds_map.get(tid)

            _drew_linearized = False
            if show_linearized_protein and cds_bounds is not None:
                try:
                    prot_len = self.protein_data.get_protein_length(tid)
                    if prot_len and prot_len > 0:
                        self._draw_linearized_protein_track(
                            ax, tid, exons, strand, cds_bounds, mapper,
                            y_center, doms_aa, id2color, exon_color, prot_len,
                        )
                        _drew_linearized = True
                except Exception:
                    pass

            if _drew_linearized:
                continue

            # Non-coding / untranslated transcript: show faint placeholder
            if show_linearized_protein and not _drew_linearized:
                _nc_y = y_center + h_exon * 1.25
                ax.add_line(mpl.lines.Line2D(
                    [mapper(int(g0)), mapper(int(g1))], [_nc_y, _nc_y],
                    color="#bbbbbb", linewidth=1.2, linestyle=(0, (4, 3)), zorder=1,
                ))
                ax.text(
                    (mapper(int(g0)) + mapper(int(g1))) / 2,
                    _nc_y + 0.022, "non-coding",
                    ha="center", va="bottom",
                    fontsize=11, color="#999999", style="italic",
                    fontweight="medium",
                )

            # ── Legacy genomic-space domain track (fallback) ──────────────────
            if not doms_aa:
                continue

            exon_bounds = sorted({min(a, b) for a, b in exons} | {max(a, b) for a, b in exons})
            dom_intervals = self.protein_data.domains_aa_to_genomic(tid, doms_aa)
            y_dom = y_center + h_exon * 1.15

            ax.add_line(mpl.lines.Line2D(
                [mapper(int(g0)), mapper(int(g1))], [y_dom],
                color="#dddddd", linewidth=2.0, zorder=1,
            ))

            if show_aa_ticks:
                try:
                    _plen = self.protein_data.get_protein_length(tid)
                    if _plen and _plen > 0 and cds_bounds:
                        _step = 25 if _plen <= 100 else 50 if _plen <= 300 else 100 if _plen <= 600 else 200
                        _ticks = list(range(1, _plen + 1, _step))
                        if _plen not in _ticks:
                            _ticks.append(_plen)
                        for _aa in _ticks:
                            _gpos = self.protein_data.aa_to_genomic(tid, _aa)
                            if _gpos is not None:
                                _xt = mapper(int(_gpos))
                                ax.add_line(mpl.lines.Line2D(
                                    [_xt, _xt],
                                    [y_dom - dom_height * 0.7, y_dom - dom_height * 0.4],
                                    color="#888888", linewidth=0.6, zorder=2,
                                ))
                                ax.text(_xt, y_dom - dom_height * 0.7 - 0.01, str(_aa),
                                        ha="center", va="top", fontsize=6, color="#666666")
                except Exception:
                    pass

            for dom in doms_aa:
                dom_id = dom["name"]
                intervals = dom_intervals.get(dom_id, [])
                if not intervals:
                    continue
                col = id2color.get(dom_id, "#444444")
                for s, e in intervals:
                    xs, xe = mapper(int(s)), mapper(int(e))
                    width = xe - xs
                    rounding = min(min(width, dom_height) * 0.3, 0.02)
                    ax.add_patch(FancyBboxPatch(
                        (xs, y_dom - dom_height / 2), width, dom_height,
                        boxstyle=f"round,pad=0,rounding_size={rounding}",
                        facecolor=col, edgecolor="black", linewidth=0.6,
                        alpha=0.95, zorder=3,
                    ))
                    for b in exon_bounds:
                        if s < b < e:
                            xb = mapper(int(b))
                            ax.add_line(mpl.lines.Line2D(
                                [xb, xb],
                                [y_dom - dom_height / 2, y_dom + dom_height / 2],
                                color="white", linewidth=0.8, zorder=4,
                            ))
        
        # Draw top ruler + gene label
        contig = self._contig_from_transcripts(tids_layout)
        gene_label = self._gene_label_from_transcripts(tids_layout)
        self._draw_top_ruler(
            ax,
            mapper,
            g0,
            g1,
            yref=ycs[0] + h_exon * 3.8,
            contig=contig,
            gene_label=gene_label,
        )
        
        # Create legend with rounded patches (DESC + ID format)
        def _short(s: str, n: int = 38) -> str:
            s = (s or "").strip()
            return s if len(s) <= n else (s[: n - 1] + "…")

        import re as _re
        legend_handles = []
        for dom_id in uniq_dom_ids:
            desc = id2label.get(dom_id) or dom_id
            # Clean up verbose "Mapped using GIFTS DB (UniProt X, ...)" labels
            if "Mapped using GIFTS DB" in desc or dom_id.startswith("AF-"):
                _m = _re.search(r'UniProt\s+(\w+)', desc)
                label = f"UniProt {_m.group(1)} (AlphaFold)" if _m else f"AlphaFold ({dom_id})"
            elif dom_id in desc or len(desc) <= 4:
                label = _short(desc)
            else:
                label = f"{_short(desc, 32)} ({dom_id})" 
            # Use FancyBboxPatch for legend with pill shape
            legend_handles.append(
                FancyBboxPatch(
                    (0, 0), 1, 1,
                    boxstyle="round,pad=0,rounding_size=0.5",
                    facecolor=id2color[dom_id],
                    edgecolor="black",
                    linewidth=0.5,
                    label=label,
                )
            )
        
        if legend_handles:
            if custom_domains:
                legend_title = "Domains (custom)"
            elif provider.lower() == "interpro":
                legend_title = "Domains (canonical only)"
            else:
                legend_title = "Domains"
            
            fig.legend(
                handles=legend_handles,
                loc="center left",
                bbox_to_anchor=(0.99, 0.5),
                frameon=False,
                title=legend_title,
                fontsize=11,
                title_fontsize=12,
            )
        
        fig.tight_layout()
        return fig
    
    def draw_gene_with_protein_domains(
        self,
        gene: str,
        adata,
        *,
        top_n: Optional[int] = None,
        provider: str = "ensembl",
        max_domains: int = 12,
        id_prefixes: Optional[List[str]] = None,
        sources: Optional[List[str]] = None,
        fig_width: float = 18.0,
        palette_name: str = "Dark2",
        custom_domains: Optional[Dict[str, List[Dict]]] = None,
        show_aa_ticks: bool = True,
        show_linearized_protein: bool = True,
    ):
        """Draw top N transcripts for a gene with protein domains.
        
        Parameters
        ----------
        gene : str
            Gene name or gene ID to visualize
        adata : AnnData
            AnnData object containing transcript expression data
        top_n : int, optional
            Number of top expressed transcripts to show
            If None, shows all transcripts for the gene
        provider : str, default "ensembl"
            Domain provider ("ensembl" or "interpro")
            Note: InterPro only annotates canonical transcripts
        max_domains : int, default 12
            Maximum domains to show per transcript
        id_prefixes : List[str], optional
            Filter domains by ID prefix (e.g., ["PF"] for Pfam)
        sources : List[str], optional
            Filter domains by source database
        fig_width : float, default 18.0
            Figure width in inches
        palette_name : str, default "Dark2"
            Seaborn color palette name for protein domains
        custom_domains : Dict[str, List[Dict]], optional
            Custom domain annotations
        show_aa_ticks : bool, default True
            Show amino acid position ticks on domain track
        
        Returns
        -------
        fig : matplotlib.figure.Figure
            The generated figure
        """
        import numpy as np
        from scipy.sparse import issparse
        
        # Helper to strip version from transcript IDs
        def _versionless(tid: str) -> str:
            """Strip .version suffix from Ensembl IDs."""
            if "." in tid:
                base, _, tail = tid.rpartition(".")
                if tail.replace("v", "").isdigit():  # Handle both .1 and .v1
                    return base
            return tid
        
        # Try to get transcripts by gene_id first, then gene_name
        all_tids = self.transcript_data.get_transcripts_by_gene_id(gene)
        if not all_tids:
            all_tids = self.transcript_data.get_transcripts_by_gene_name(gene)
        
        if not all_tids:
            raise ValueError(f"No transcripts found for gene: {gene}")
        
        # If top_n specified, select top N by expression
        if top_n is not None:
            # Build mapping from versionless ID to full IDs
            gene_base_to_full = {}
            for tid in all_tids:
                base = _versionless(tid)
                gene_base_to_full.setdefault(base, []).append(tid)
            
            adata_base_to_full = {}
            for tid in adata.var_names:
                base = _versionless(str(tid))
                adata_base_to_full.setdefault(base, []).append(str(tid))
            
            # Match gene transcripts to adata transcripts (version-insensitive)
            tid_indices = []
            matched_tids = []
            
            for gene_tid in all_tids:
                gene_base = _versionless(gene_tid)
                
                # Try exact match first
                if gene_tid in adata.var_names:
                    tid_indices.append(adata.var_names.get_loc(gene_tid))
                    matched_tids.append(gene_tid)
                # Try versionless match
                elif gene_base in adata_base_to_full:
                    # Use the first match in adata
                    adata_tid = adata_base_to_full[gene_base][0]
                    tid_indices.append(adata.var_names.get_loc(adata_tid))
                    matched_tids.append(gene_tid)
            
            if not tid_indices:
                raise ValueError(
                    f"None of the gene's {len(all_tids)} transcripts found in adata.var_names.\n"
                    f"Gene transcripts: {all_tids[:5]}\n"
                    f"Example adata transcripts: {list(adata.var_names[:5])}"
                )
            
            # Calculate mean expression for matched transcripts
            X = adata.X
            if issparse(X):
                means = np.asarray(X[:, tid_indices].mean(axis=0)).ravel()
            else:
                means = X[:, tid_indices].mean(axis=0)
                if hasattr(means, 'ravel'):
                    means = means.ravel()
            
            # Get top N by expression
            k = min(top_n, len(means))
            top_indices = np.argsort(means)[::-1][:k]
            transcript_ids = [matched_tids[i] for i in top_indices]
        else:
            # No filtering, use all transcripts
            transcript_ids = all_tids
        
        # Draw using the main method
        return self.draw_transcripts_with_protein_domains(
            transcript_ids,
            provider=provider,
            max_domains=max_domains,
            id_prefixes=id_prefixes,
            sources=sources,
            fig_width=fig_width,
            palette_name=palette_name,
            custom_domains=custom_domains,
            show_aa_ticks=show_aa_ticks,
            show_linearized_protein=show_linearized_protein,
        )

## Examples

Demonstrate protein domain visualization on transcripts.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()